# Hybrid RAG: leave policy assistant

A question-answering pipeline over the company's leave policies (`../Leave_Policy`, 12 PDFs, 2015–2026), using
**hybrid retrieval**: every question runs through two searches whose results are merged.

| Retriever | Finds chunks by | Strong at | Weak at |
|---|---|---|---|
| **Keyword (BM25)** | the question's exact words, with rare words weighted higher | years (`2019`), ids (`HR-LV-2022`), exact leave names | different wording (*"look after a sick parent"* vs *"Caregiver leave"*) |
| **Dense (vectors in Pinecone)** | meaning | paraphrases and synonyms | exact numbers and ids |

The two ranked lists are merged with **Reciprocal Rank Fusion (RRF)**. When the question names a policy year, both
searches are limited to that year (**metadata filter**), because the 12 policies are nearly identical apart from their numbers.

### Pipeline

```
INGEST   PDFs ─► pages ─► chunks ─┬─► BM25 keyword index (in memory)
                                  └─► embeddings ─► Pinecone (namespace "hybrid-rag")

QUERY    question ─► detect year ─┬─► keyword search ─┐
                                  └─► dense search ───┴─► RRF fusion ─► top-k chunks ─► Gemini ─► answer + sources
```

### Cell map
| Cell | Responsibility |
|---|---|
| 1 | Imports and logging |
| 2 | Configuration (every tunable setting in one place) |
| 3 | External clients: embedding model, Pinecone, Gemini |
| 4 | Load the PDFs |
| 5 | Chunk the pages |
| 6 | Tokenizer for keyword search |
| 7 | `KeywordRetriever` (BM25) |
| 8 | `DenseRetriever` (Pinecone): sync + search |
| 9 | Reciprocal Rank Fusion |
| 10 | `HybridRetriever`: year detection + fusion |
| 11 | LLM generation with model fallback |
| 12 | `RAGPipeline`: retrieve → prompt → answer |
| 13 | Ingestion: build both indexes |
| 14 | Query: ask questions |

Run the cells top to bottom. After that, only cell 14 needs re-running to ask new questions.

### 1. Imports and logging

**Purpose:** load the libraries and set up logging, which production code uses instead of `print` to report progress and problems.

| Import | Used for |
|---|---|
| `logging` | timestamped progress / warning messages (`logger.info`, `logger.warning`) |
| `os`, `re`, `json`, `time` | file paths and env vars, tokenising, reading the manifest, waiting for Pinecone |
| `numpy` | sorting BM25 scores quickly (`np.argsort`) |
| `PdfReader` (pypdf) | reading PDF pages |
| `load_dotenv`, `find_dotenv` | loading API keys from `.env` |
| `RecursiveCharacterTextSplitter` | chunking |
| `BM25Okapi` (rank_bm25) | the standard BM25 keyword-ranking implementation |
| `SentenceTransformer` | the embedding model |
| `Pinecone`, `ServerlessSpec` | the vector database |
| `genai`, `types`, `genai_errors` | Gemini client, request settings, and its error classes (for the fallback in cell 11) |

**Logging setup**
- `logging.basicConfig(level=logging.WARNING, ..., force=True)`: libraries (Pinecone, Hugging Face, httpx) only show **warnings and errors**,
  so their per-request messages don't flood the output. `force=True` is needed in Jupyter, which already has a handler that would otherwise swallow the messages.
- `logger = logging.getLogger("hybrid_rag")` at level `INFO`: this pipeline's own progress messages are shown.
- The format `HH:MM:SS LEVEL logger-name: message` shows when each step ran and which component logged it.

**Output:** nothing.

In [ ]:
import json
import logging
import os
import re
import time

import numpy as np
from dotenv import find_dotenv, load_dotenv
from google import genai
from google.genai import errors as genai_errors
from google.genai import types
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone, ServerlessSpec
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer


logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(name)s: %(message)s",
                    datefmt="%H:%M:%S", force=True)
logger = logging.getLogger("hybrid_rag")
logger.setLevel(logging.INFO)

### 2. Configuration

**Purpose:** keep every setting in one place, so tuning the system never means hunting through the code.

| Setting | Value | Meaning |
|---|---|---|
| `DATA_FOLDER` | `../Leave_Policy` | input PDFs + `manifest.json` (one folder up from this notebook) |
| `EMBEDDING_MODEL` / `EMBEDDING_DIM` | MiniLM / 384 | the embedding model and its vector size. The Pinecone index must have the same dimension |
| `PINECONE_NAMESPACE` | `hybrid-rag` | this pipeline's own section of the shared Pinecone index |
| `PINECONE_CLOUD` / `PINECONE_REGION` | aws / us-east-1 | used only if the index has to be created |
| `CHUNK_SIZE` / `CHUNK_OVERLAP` | 800 / 100 characters | chunk length, and how much neighbouring chunks share |
| `CANDIDATES_PER_RETRIEVER` | 20 | how many results each search returns before fusion. More = better recall, slightly slower |
| `TOP_K` | 5 | how many fused chunks are sent to the LLM |
| `RRF_K` | 60 | RRF smoothing constant (60 is the standard value) |
| `RRF_WEIGHTS` | keyword 1.0, dense 1.0 | raise one to trust that search more |
| `BM25_K1` / `BM25_B` | 1.5 / 0.75 | BM25 term-saturation and length-normalisation (standard values) |
| `LLM_MODELS` | primary, fallback | Gemini models tried in order. The next one is used if the first is overloaded |
| `LLM_TEMPERATURE` | 0 | 0 = the most deterministic, factual answers. Higher values give more varied wording |
| `INDEX_READY_TIMEOUT_S` | 60 | how long to wait for Pinecone to make new vectors searchable |

`load_dotenv(find_dotenv(usecwd=True))` loads `.env` by searching from this folder upwards, so the project-root `.env` is found.
The three keys it must contain: `PineCone_API_Key`, `Pinecone_Index_Name`, `GEMINI_API_KEY`.

**Output:** nothing.

In [ ]:
load_dotenv(find_dotenv(usecwd=True))

DATA_FOLDER = "../Leave_Policy"
MANIFEST_FILE = "manifest.json"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

PINECONE_API_KEY = os.getenv("PineCone_API_Key")
PINECONE_INDEX = os.getenv("Pinecone_Index_Name")
PINECONE_NAMESPACE = "hybrid-rag"
PINECONE_CLOUD, PINECONE_REGION = "aws", "us-east-1"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 100

CANDIDATES_PER_RETRIEVER = 20
TOP_K = 5
RRF_K = 60
RRF_WEIGHTS = {"keyword": 1.0, "dense": 1.0}
BM25_K1, BM25_B = 1.5, 0.75

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LLM_MODELS = ["gemini-3.5-flash", "gemini-3.5-flash-lite"]
LLM_TEMPERATURE = 0.0

INDEX_READY_TIMEOUT_S = 60

### 3. External clients

**Purpose:** connect to the three external components once, and fail early with a clear message if a key is missing.

1. **Key check:** if any of the three `.env` values is empty, raise a `RuntimeError` naming the missing ones,
   instead of failing later with a confusing error inside a library.
2. **Embedding model:** `SentenceTransformer(EMBEDDING_MODEL)`. Downloaded on first use (about 90 MB), then loaded from the local cache.
3. **Pinecone:** connects, and creates the index only if it doesn't exist yet (`dimension` must match the embedding size, `metric="cosine"`).
   - If the index exists with a **different** dimension, it raises an error straight away, because every upsert would fail otherwise.
4. **Gemini:** `genai.Client(api_key=...)`.

**Output:** one log line confirming the index and namespace.

In [ ]:
missing = [name for name, value in {"PineCone_API_Key": PINECONE_API_KEY,
                                    "Pinecone_Index_Name": PINECONE_INDEX,
                                    "GEMINI_API_KEY": GEMINI_API_KEY}.items() if not value]
if missing:
    raise RuntimeError(f"Missing in .env: {', '.join(missing)}")

embedder = SentenceTransformer(EMBEDDING_MODEL)

pc = Pinecone(api_key=PINECONE_API_KEY)
if PINECONE_INDEX not in pc.list_indexes().names():
    logger.info("Creating Pinecone index %s", PINECONE_INDEX)
    pc.create_index(name=PINECONE_INDEX, dimension=EMBEDDING_DIM, metric="cosine",
                    spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION))
index_dim = pc.describe_index(PINECONE_INDEX).dimension
if index_dim != EMBEDDING_DIM:
    raise RuntimeError(f"Index {PINECONE_INDEX} has dimension {index_dim}, embedding model needs {EMBEDDING_DIM}")
pinecone_index = pc.Index(PINECONE_INDEX)

llm_client = genai.Client(api_key=GEMINI_API_KEY)

logger.info("Clients ready: index=%s namespace=%s", PINECONE_INDEX, PINECONE_NAMESPACE)

### 4. Load the documents

**Purpose:** read every policy PDF listed in the manifest into a list of pages, each carrying its metadata.

**`load_policy_pages(folder)`**
1. Reads `manifest.json`. It lists each policy's `year`, `file` and `doc_id` (e.g. `HR-LV-2019`).
2. For each policy, opens the PDF with `PdfReader` and extracts the text page by page.
   Pages are kept separate so every answer can cite the page number.
3. **Robustness:**
   - a file listed in the manifest but **missing** on disk is logged as a warning and skipped, so one bad file doesn't stop ingestion
   - an empty page (`extract_text()` returns nothing) is skipped
4. Returns `(manifest, pages)`, where each page is `{"year", "doc_id", "file", "page", "text"}`.

**Output:** nothing (the function is called in cell 13).

In [ ]:
def load_policy_pages(folder):
    with open(os.path.join(folder, MANIFEST_FILE), encoding="utf-8") as f:
        manifest = json.load(f)

    pages = []
    for doc in manifest:
        path = os.path.join(folder, doc["file"])
        if not os.path.exists(path):
            logger.warning("Missing file listed in manifest, skipped: %s", path)
            continue
        for page_no, page in enumerate(PdfReader(path).pages, start=1):
            text = (page.extract_text() or "").strip()
            if text:
                pages.append({"year": doc["year"], "doc_id": doc["doc_id"], "file": doc["file"],
                              "page": page_no, "text": text})
    return manifest, pages

### 5. Chunk the pages

**Purpose:** split each page into chunks small enough to search precisely.

**`chunk_pages(pages)`**
- **Splitter:** `RecursiveCharacterTextSplitter` cuts at paragraph breaks first, then line breaks, then spaces, so chunks
  end at natural boundaries. Size `CHUNK_SIZE`, overlap `CHUNK_OVERLAP`.
- **Context header:** every chunk's text starts with `Leave Policy 2019 (HR-LV-2019), page 2: ...`.
  A chunk from the middle of a page may not mention its year otherwise, and then neither search could tell 2019 from 2020.
- **Stable ids:** `2019-p2-c1` = policy 2019, page 2, chunk 1. The same input always gives the same ids, so re-ingesting
  **overwrites** vectors instead of duplicating them, and cell 8 can detect stale ones.

Each chunk is `{"id", "text", "year", "doc_id", "page"}`.

**Output:** nothing (called in cell 13).

In [ ]:
def chunk_pages(pages):
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
                                              separators=["\n\n", "\n", " ", ""])
    chunks = []
    for p in pages:
        for i, piece in enumerate(splitter.split_text(p["text"])):
            chunks.append({
                "id": f"{p['year']}-p{p['page']}-c{i}",
                "text": f"Leave Policy {p['year']} ({p['doc_id']}), page {p['page']}: {piece}",
                "year": p["year"],
                "doc_id": p["doc_id"],
                "page": p["page"],
            })
    return chunks

### 6. Tokenizer for keyword search

**Purpose:** turn text into the list of words BM25 compares. The **same** function is used for the chunks and the questions,
so both are split identically. Different tokenizers would make matches silently fail.

**`tokenize(text)`**
1. `lower()`: case-insensitive matching (`Sick` = `sick`).
2. `re.findall(r"[a-z0-9]+", ...)`: keeps runs of letters/digits, so `HR-LV-2019` becomes `hr`, `lv`, `2019` and a question saying only *2019* still matches.
3. Drops `STOPWORDS`, common question words (*what, how, many, the, of ...*) that would only add noise.

**Output:** nothing.

In [ ]:
STOPWORDS = frozenset({
    "a", "an", "the", "of", "in", "on", "for", "to", "and", "or", "is", "are", "was", "were", "be", "been",
    "what", "which", "who", "how", "many", "much", "does", "do", "did", "can", "i", "my", "me", "we", "our",
    "it", "its", "this", "that", "by", "with", "at", "as", "from", "per", "any", "there", "get", "got",
})


def tokenize(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS]

### 7. `KeywordRetriever` (BM25)

**Purpose:** exact-word search over the chunks.

**How BM25 scores a chunk:** for each question word found in the chunk it adds
`idf × (tf × (k1+1)) / (tf + k1 × (1 − b + b × len/avg_len))`
- **idf** (inverse document frequency): rare words (*2019*, *fertility*) count a lot; words in almost every chunk (*leave*) count very little.
- **tf** (term frequency) with **`k1`** saturation: repeating a word helps, but less each time.
- **`b`** length normalisation: long chunks don't win just by containing more words.

`BM25Okapi` from `rank_bm25` is the standard implementation of this formula.

**The class**
- `__init__(chunks)`: tokenizes every chunk and builds the BM25 index **in memory**. It is rebuilt at every start-up
  (milliseconds for this corpus). A larger system would use a search engine such as Elasticsearch or OpenSearch instead.
- `search(query, k, year=None)`:
  1. tokenizes the query, and returns nothing if no searchable words remain (e.g. the query was only stopwords)
  2. scores **all** chunks, then walks them from highest to lowest score (`np.argsort(...)[::-1]`)
  3. stops at the first score ≤ 0 (no shared words), skips chunks from other years when `year` is set, and stops after `k` results
  4. returns `[(chunk_id, score), ...]`, best first

**Output:** nothing.

In [ ]:
class KeywordRetriever:
    def __init__(self, chunks):
        self.chunk_ids = [c["id"] for c in chunks]
        self.years = [c["year"] for c in chunks]
        self.bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks], k1=BM25_K1, b=BM25_B)

    def search(self, query, k, year=None):
        tokens = tokenize(query)
        if not tokens:
            return []
        scores = self.bm25.get_scores(tokens)
        results = []
        for i in np.argsort(scores)[::-1]:
            if scores[i] <= 0:
                break
            if year is not None and self.years[i] != year:
                continue
            results.append((self.chunk_ids[i], float(scores[i])))
            if len(results) == k:
                break
        return results

### 8. `DenseRetriever` (Pinecone)

**Purpose:** meaning-based search, plus keeping Pinecone in sync with the current chunks.

**`sync(chunks)`**: makes the namespace contain **exactly** the current chunks
1. **Embed:** `embedder.encode(..., batch_size=64)` turns all chunk texts into vectors, 64 at a time.
2. **Upsert** in batches of 100 (Pinecone's recommended request size). Metadata stores `year`, `doc_id`, `page`, `text`.
   Because ids are stable, existing vectors are overwritten, not duplicated.
3. **Delete stale vectors:** `index.list(namespace=...)` pages through every id currently stored (each item's `.id`). Any id that is **not**
   in this run's chunks (e.g. left over after changing `CHUNK_SIZE`, or from a deleted PDF) is deleted, in batches of 1000.
   This is safer than wiping the namespace: the index stays searchable during re-ingestion.
4. **Wait until searchable:** Pinecone indexes new writes asynchronously, so it polls `describe_index_stats()` until the
   namespace count equals the chunk count (up to `INDEX_READY_TIMEOUT_S`). If the timeout is reached it logs a warning.

**`search(query, k, year=None)`**
- Embeds the query with the **same** model and asks Pinecone for the `k` most similar vectors.
- `year` set: `filter={"year": {"$eq": year}}` limits the search to that policy on the Pinecone side.
- Returns `[(chunk_id, cosine_score), ...]`, best first.

**Output:** nothing.

In [ ]:
class DenseRetriever:
    def __init__(self, index, embedder, namespace):
        self.index = index
        self.embedder = embedder
        self.namespace = namespace

    def sync(self, chunks):
        vectors = self.embedder.encode([c["text"] for c in chunks], batch_size=64)
        records = [
            {"id": c["id"], "values": v.tolist(),
             "metadata": {"year": c["year"], "doc_id": c["doc_id"], "page": c["page"], "text": c["text"]}}
            for c, v in zip(chunks, vectors)
        ]
        for start in range(0, len(records), 100):
            self.index.upsert(vectors=records[start:start + 100], namespace=self.namespace)

        current_ids = {c["id"] for c in chunks}
        stored_ids = [getattr(item, "id", item) for page in self.index.list(namespace=self.namespace) for item in page]
        stale = [vid for vid in stored_ids if vid not in current_ids]
        for start in range(0, len(stale), 1000):
            self.index.delete(ids=stale[start:start + 1000], namespace=self.namespace)

        deadline = time.time() + INDEX_READY_TIMEOUT_S
        while self._count() != len(chunks) and time.time() < deadline:
            time.sleep(2)
        if self._count() != len(chunks):
            logger.warning("Pinecone not fully synced yet: %d of %d vectors visible", self._count(), len(chunks))
        logger.info("Dense index synced: %d vectors upserted, %d stale removed", len(records), len(stale))

    def _count(self):
        ns = self.index.describe_index_stats().namespaces.get(self.namespace)
        return ns.vector_count if ns else 0

    def search(self, query, k, year=None):
        res = self.index.query(
            vector=self.embedder.encode(query).tolist(),
            top_k=k,
            namespace=self.namespace,
            filter={"year": {"$eq": year}} if year is not None else None,
        )
        return [(m.id, m.score) for m in res.matches]

### 9. Reciprocal Rank Fusion (RRF)

**Purpose:** merge several ranked result lists into one ranking.

**Why ranks and not scores?** BM25 scores (roughly 0–30) and cosine scores (0–1) are on different scales. Adding them would
let BM25 dominate, and normalising them is fragile. RRF uses only each chunk's **position** in each list:

```
RRF(chunk) = Σ over lists  weight / (RRF_K + rank)
```
- Rank 1 → `1/61`, rank 10 → `1/70`, absent from a list → 0.
- A chunk ranked well by **both** searches collects two contributions and rises to the top.
- `RRF_K = 60` damps the gap between the very top ranks, so one list's #1 can't override strong agreement from both.

**`reciprocal_rank_fusion(ranked_lists)`**
- Input: `{"keyword": [(id, score), ...], "dense": [(id, score), ...]}`.
- Weights come from `RRF_WEIGHTS`; a list missing from the config gets weight 1.0.
- Returns `[(chunk_id, rrf_score), ...]`, highest first.

**Output:** nothing.

In [ ]:
def reciprocal_rank_fusion(ranked_lists):
    fused = {}
    for name, results in ranked_lists.items():
        weight = RRF_WEIGHTS.get(name, 1.0)
        for rank, (chunk_id, _score) in enumerate(results, start=1):
            fused[chunk_id] = fused.get(chunk_id, 0.0) + weight / (RRF_K + rank)
    return sorted(fused.items(), key=lambda pair: pair[1], reverse=True)

### 10. `HybridRetriever`: year detection + both searches + fusion

**Purpose:** the single entry point for retrieval. Give it a question; get back the best chunks.

**`detect_year(question)`**
- Finds 4-digit years with `\b(20\d{2})\b`, which also matches the year inside `HR-LV-2022` (the `-` counts as a word boundary).
- Returns the first year that is an actual policy year (from the manifest), otherwise `None`. So *"in 2030"* doesn't filter everything out.

**Why filter by year?** The 12 policies share almost the same text. Without a filter, chunks from other years with similar
wording push the right year's chunk down the list. With the filter, both searches only compare chunks from the right policy.

**`retrieve(question, top_k=TOP_K)`**
1. `year = detect_year(question)`
2. Runs the keyword and dense searches, each returning `CANDIDATES_PER_RETRIEVER` results, both limited to `year` if one was found.
3. Fuses them with RRF (cell 9) and keeps the top `top_k`.
4. Returns `(year, hits)`. Each hit is the full chunk plus its `score` (RRF) and `found_by` (which searches returned it:
   `keyword`, `dense` or both), which is useful for debugging retrieval quality.

**Output:** nothing.

In [ ]:
class HybridRetriever:
    def __init__(self, chunks, keyword, dense, policy_years):
        self.chunk_by_id = {c["id"]: c for c in chunks}
        self.keyword = keyword
        self.dense = dense
        self.policy_years = set(policy_years)

    def detect_year(self, question):
        for year in re.findall(r"\b(20\d{2})\b", question):
            if int(year) in self.policy_years:
                return int(year)
        return None

    def retrieve(self, question, top_k=TOP_K):
        year = self.detect_year(question)
        ranked = {
            "keyword": self.keyword.search(question, CANDIDATES_PER_RETRIEVER, year=year),
            "dense": self.dense.search(question, CANDIDATES_PER_RETRIEVER, year=year),
        }
        found_by = {}
        for name, results in ranked.items():
            for chunk_id, _ in results:
                found_by.setdefault(chunk_id, []).append(name)

        hits = []
        for chunk_id, score in reciprocal_rank_fusion(ranked):
            if chunk_id in self.chunk_by_id:
                hits.append({**self.chunk_by_id[chunk_id], "score": score, "found_by": found_by[chunk_id]})
            if len(hits) == top_k:
                break
        return year, hits

### 11. LLM generation (Gemini, with model fallback)

**Purpose:** send the prompt to Gemini and return the answer, recovering from temporary outages.

**`SYSTEM_PROMPT`**: standing rules, sent as Gemini's `system_instruction` (kept apart from the question):

| Rule | Why |
|---|---|
| Use only the sources | no answers from general knowledge |
| Each year has its own policy: use the year asked; if none, the latest year in the sources, and say which | 12 versions of the same policy exist; mixing years gives wrong numbers |
| Cite `[Source n]`, keep numbers exact | every fact can be checked |
| Say so if the answer isn't in the sources | no invented answers |

**`generate(prompt)`**
- Calls Gemini with `temperature=LLM_TEMPERATURE` (0 = deterministic, factual).
- **Fallback:** tries each model in `LLM_MODELS` in order. On a retryable error (429 rate limit, 500, 503 overloaded) it logs a warning and
  moves to the next model. Other errors (e.g. 400/403, a bad key) are raised immediately, because retrying won't fix them.
- If the reply is empty (blocked by safety filters), it returns a fixed message instead of `None`.
- Returns `(answer_text, model_used)`.

**Output:** nothing.

In [ ]:
SYSTEM_PROMPT = """You answer employees' questions about Acme Technologies' leave policies, using only the numbered sources provided.
The company has a separate policy for each year (2015-2026), and the numbers change between years.
- Use the policy of the year the question asks about. If no year is asked, use the latest year among the sources and say which year.
- Cite the sources you used, e.g. [Source 2]. Do not change any numbers.
- If the sources don't contain the answer, say that you don't have that information."""

RETRYABLE_STATUS = {429, 500, 503}


def generate(prompt):
    config = types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, temperature=LLM_TEMPERATURE)
    last_error = None
    for model_name in LLM_MODELS:
        try:
            response = llm_client.models.generate_content(model=model_name, contents=prompt, config=config)
            return (response.text or "The model returned no answer for this question."), model_name
        except genai_errors.APIError as e:
            if e.code not in RETRYABLE_STATUS:
                raise
            logger.warning("%s unavailable (%s), trying next model", model_name, e.code)
            last_error = e
    raise last_error

### 12. `RAGPipeline`: retrieve → build prompt → answer

**Purpose:** tie retrieval and generation together behind one call: `pipeline.ask(question)`.

**`ask(question)`**
1. `retriever.retrieve(question)` returns the detected year and the top chunks (cell 10).
2. **No hits:** returns a "not found" answer without calling the LLM (saves cost, avoids a made-up answer).
3. **Build the prompt:** each chunk is labelled `[Source n] Leave Policy <year>, page <p>`, followed by the question.
4. `generate(prompt)` returns the answer and the model that produced it (cell 11).
5. Returns a **dict**, not printed text, so the same pipeline can feed a notebook, a web API or a chat UI:

| Key | Content |
|---|---|
| `question` | the question asked |
| `year_filter` | the policy year detected, or `None` |
| `answer` | Gemini's answer, with `[Source n]` citations |
| `model` | which Gemini model answered |
| `sources` | for each source: number, year, page, chunk id, RRF score, which searches found it |
| `latency_s` | total time taken, in seconds |

**`format_result(result)`**: turns that dict into readable text for the notebook.

**Output:** nothing.

In [ ]:
class RAGPipeline:
    def __init__(self, retriever):
        self.retriever = retriever

    def ask(self, question):
        started = time.perf_counter()
        year, hits = self.retriever.retrieve(question)

        if not hits:
            answer, model = "I couldn't find anything about that in the leave policies.", None
        else:
            context = "\n\n".join(f"[Source {n}] Leave Policy {h['year']}, page {h['page']}:\n{h['text']}"
                                  for n, h in enumerate(hits, start=1))
            answer, model = generate(f"{context}\n\nQuestion: {question}")

        return {
            "question": question,
            "year_filter": year,
            "answer": answer,
            "model": model,
            "sources": [{"source": n, "year": h["year"], "page": h["page"], "chunk_id": h["id"],
                         "score": round(h["score"], 4), "found_by": h["found_by"]}
                        for n, h in enumerate(hits, start=1)],
            "latency_s": round(time.perf_counter() - started, 2),
        }


def format_result(result):
    lines = [f"Q: {result['question']}",
             f"   year filter: {result['year_filter'] or 'none'} | model: {result['model']} | {result['latency_s']}s",
             "", result["answer"].strip(), "", "Sources:"]
    lines += [f"  [Source {s['source']}] Leave Policy {s['year']}, page {s['page']} "
              f"({s['chunk_id']}, RRF {s['score']}, found by {'+'.join(s['found_by'])})"
              for s in result["sources"]]
    return "\n".join(lines)

### 13. Ingestion: build both indexes

**Purpose:** run the ingest side of the pipeline once at start-up. Re-run it only when the PDFs or chunk settings change.

1. `load_policy_pages(DATA_FOLDER)` reads the manifest and PDFs (cell 4).
2. `chunk_pages(pages)` splits them into chunks (cell 5).
3. `KeywordRetriever(chunks)` builds the BM25 index in memory (cell 7).
4. `DenseRetriever(...).sync(chunks)` embeds, upserts, removes stale vectors and waits until Pinecone is searchable (cell 8).
5. Wires everything into `HybridRetriever` (cell 10) and `RAGPipeline` (cell 12).

**Output:** log lines with the number of policies, pages, chunks and vectors synced, and the total time.

In [ ]:
started = time.perf_counter()

manifest, pages = load_policy_pages(DATA_FOLDER)
chunks = chunk_pages(pages)
logger.info("Loaded %d policies, %d pages, %d chunks", len(manifest), len(pages), len(chunks))

keyword_retriever = KeywordRetriever(chunks)
dense_retriever = DenseRetriever(pinecone_index, embedder, PINECONE_NAMESPACE)
dense_retriever.sync(chunks)

retriever = HybridRetriever(chunks, keyword_retriever, dense_retriever, policy_years=[d["year"] for d in manifest])
pipeline = RAGPipeline(retriever)

logger.info("Ingestion finished in %.1fs", time.perf_counter() - started)

### 14. Ask questions

**Purpose:** query the pipeline. Edit the `questions` list (or call `pipeline.ask("...")` directly) and re-run only this cell.

The four sample questions each exercise a different part of the hybrid design:

| Question | What it relies on | Expected answer |
|---|---|---|
| Earned leave in **2019** | keyword match on the year + year filter | 18 days |
| *Look after a sick family member* | dense search (the policy says *"Caregiver leave"*, a different wording) | caregiver leave, 5 days per year |
| Casual leave in **HR-LV-2022** | keyword match on the document id + year filter | 10 days |
| When was **fertility treatment leave** introduced | keyword match on the exact leave name across years | 2026, 5 days per year |

**Reading the sources:** `found by` lists which searches had the chunk among their top `CANDIDATES_PER_RETRIEVER` results.
`keyword+dense` means both agreed. RRF ranks such chunks highest, which is why they usually fill the top 5. If a chunk shows
only `keyword` or only `dense`, that's a chunk one search alone would have missed. The `year filter` line shows whether the
search was limited to one policy year.

In [ ]:
questions = [
    "How many days of earned leave did employees get in 2019?",
    "Can I take time off to look after a sick family member?",
    "What is the casual leave entitlement in policy HR-LV-2022?",
    "When was fertility treatment leave introduced and how many days does it give?",
]

for question in questions:
    print(format_result(pipeline.ask(question)))
    print("-" * 100)